<a href="https://colab.research.google.com/github/lillianc05-debug/Web-Scraping-Course-Schedule/blob/main/Assignment_2_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
import bs4
from bs4 import BeautifulSoup
import json

In [ ]:
Systems_Analysis = "https://bit.vt.edu/faculty/directory/romero-masters.html" #Philip Romero-Masters
Cyber_Law = "https://finance.pamplin.vt.edu/about-us/directory/malone.html" #Jason Malone
DataMgt_BusAnalytics = "https://bit.vt.edu/faculty/directory/bradberry.html" #Caleb Bradberry
Election_Security = "https://bit.vt.edu/faculty/directory/monday.html" #Justin Monday
Intermediate_Jumping = "https://sas.vt.edu/people/faculty/sheely-beth.html" #Beth Sheely
Networks_Telecom = "https://bit.vt.edu/faculty/directory/romero-masters.html" #Philip Romero-Masters
German_StudyAbroad = "https://sas.vt.edu/people/faculty/sheely-beth.html" #Beth Sheely

courses = {
    "Systems Analysis": Systems_Analysis,
    "Cyber Law": Cyber_Law,
    "Data Management and Business Analytics": DataMgt_BusAnalytics,
    "Election Security": Election_Security,
    "Intermediate Jumping": Intermediate_Jumping,
    "Networks and Telecom": Networks_Telecom,
    "German Study Abroad": German_StudyAbroad
}

In [ ]:
def scrape_vt_profile(course_name, url):
    try:
        response = requests.get(url, timeout=10)
        print(f"{course_name}: status code = {response.status_code}")

        if response.status_code != 200:
            return {
                "Course": course_name,
                "Name": "N/A",
                "Title": "N/A",
                "Office_Location": "N/A",
                "Email": "N/A"
            }

        soup = BeautifulSoup(response.text, "html.parser")

        lines = [line.strip() for line in soup.get_text("\n").split("\n") if line.strip()]

        name_tag = soup.find("h1")
        name = name_tag.get_text(strip=True) if name_tag else "N/A"

        title = "N/A"
        if name_tag:
            next_tag = name_tag.find_next(["h2", "p", "div"])
            while next_tag:
                  text = next_tag.get_text(" ", strip=True)
                  if text and text not in ["Copy page address to clipboard", "/"]:
                      title = text
                      break
                  next_tag = next_tag.find_next(["h2", "p", "div"])

        office_location = "N/A"
        if "Location:" in lines and "Contact:" in lines:
            loc_start = lines.index("Location:") + 1
            loc_end = lines.index("Contact:")
            office_parts = lines[loc_start:loc_end]
            office_location = ", ".join(office_parts) if office_parts else "N/A"

        email = "N/A"
        if "Contact:" in lines:
            contact_start = lines.index("Contact:") + 1
            for line in lines[contact_start:contact_start + 8]:
                if "@" in line:
                    email = line
                    break

        return {
            "Course": course_name,
            "Name": name,
            "Title": title,
            "Office_Location": office_location,
            "Email": email
        }

    except requests.RequestException as e:
        print(f"{course_name}: request failed -> {e}")
        return {
            "Course": course_name,
            "Name": "N/A",
            "Title": "N/A",
            "Office_Location": "N/A",
            "Email": "N/A"
        }

faculty_data = []

for course, url in courses.items():
    result = scrape_vt_profile(course, url)
    faculty_data.append(result)

df = pd.DataFrame(faculty_data)
print(df)

df.to_csv("Cunningham_sp26.csv", index=False)

with open("Cunningham_sp26.json", "w") as json_file:
    json.dump(faculty_data, json_file, indent=4)

print("\nFiles saved:")
print("Cunningham_sp26.csv")
print("Cunningham_sp26.json")

Systems Analysis: status code = 200
Cyber Law: status code = 200
Data Management and Business Analytics: status code = 200
Election Security: status code = 200
Intermediate Jumping: status code = 200
Networks and Telecom: status code = 200
German Study Abroad: status code = 200
                                   Course                   Name  \
0                        Systems Analysis  Philip Romero-Masters   
1                               Cyber Law           Jason Malone   
2  Data Management and Business Analytics        Caleb Bradberry   
3                       Election Security          Justin Monday   
4                    Intermediate Jumping            Beth Sheely   
5                    Networks and Telecom  Philip Romero-Masters   
6                     German Study Abroad            Beth Sheely   

                                      Title  \
0            Assistant Collegiate Professor   
1           Associate Professor of Practice   
2           Assistant Professor of 